# Lecture 11: Retrieval-Augmented Generation (RAG)

## What we will build
- a tiny local RAG system
- dense retrieval with sentence embeddings
- optional reranking
- answer generation with a small instruct model

## Constraints for this notebook
- no paid API keys
- should run on a MacBook with Apple Silicon
- short markdown, more code


In [ ]:
# @title
%pip install -q sentence-transformers transformers scikit-learn pandas


## Why do we still need RAG in 2026?

Long-context models are much better now.
Limited context window is no longer the main bottleneck for many tasks.

We still need RAG because:
- model parameters are stale the moment training ends
- we want answers grounded in private or local documents
- dumping an entire corpus into the prompt adds noise
- citations, access control, freshness, and deletions live outside the LLM
- retrieving 3 useful chunks is still cheaper and more reliable than sending 300 pages


## LLM limitations without RAG

Even a strong LLM can struggle with:
- organization-specific facts
- recently updated information
- traceable answers with sources
- user or tenant specific permissions
- knowing when it should say "I do not have evidence"


## Production RAG in one line

`source documents -> chunking -> embeddings -> vector index -> retrieval -> reranking -> prompt assembly -> generation -> evaluation`

Retrieval is only one piece.
In production, ingestion, metadata, permissions, freshness, and evaluation matter just as much.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from IPython.display import display
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "mps" else torch.float32

print("device:", device)


## A tiny knowledge base

We will create a synthetic corpus that looks like internal documentation for a production RAG system.
This gives us facts that the base model is unlikely to know on its own.


In [ ]:
documents = [
    {
        "doc_id": "DOC-001",
        "title": "System overview",
        "team": "platform",
        "updated_at": "2026-02-14",
        "text": (
            "Atlas Assist is a production RAG system used inside an enterprise support portal. "
            "The online path has six stages: query understanding, retrieval, reranking, prompt assembly, generation, and answer validation. "
            "The assistant must always return citations and should abstain when the retrieved evidence is weak. "
            "Only the final three passages are sent to the generator even though the retriever may inspect a much larger candidate set."
        ),
    },
    {
        "doc_id": "DOC-002",
        "title": "Chunking policy",
        "team": "ingestion",
        "updated_at": "2026-02-20",
        "text": (
            "Atlas Assist chunks documents by headings first and only then applies a token budget. "
            "The target chunk size is 350 to 500 tokens with an overlap of about 50 tokens. "
            "Tables, code blocks, and bullet lists are kept intact when possible because breaking structure often hurts retrieval quality. "
            "Every chunk stores source_uri, title, section, updated_at, acl, and embedding_version as metadata."
        ),
    },
    {
        "doc_id": "DOC-003",
        "title": "Vector store notes",
        "team": "platform",
        "updated_at": "2026-02-22",
        "text": (
            "The vector database stores dense embeddings, document ids, chunk ids, and metadata filters. "
            "In production the team uses an approximate nearest neighbour index based on HNSW. "
            "A vector database is more than a NumPy matrix: it supports persistence, metadata filtering, upserts, deletes, and multi-tenant isolation. "
            "For classroom demos, an in-memory cosine similarity search is enough to explain the idea."
        ),
    },
    {
        "doc_id": "DOC-004",
        "title": "Retrieval and reranking",
        "team": "search",
        "updated_at": "2026-02-24",
        "text": (
            "The first stage retriever uses a bi-encoder embedding model and returns the top 20 semantic matches. "
            "Those candidates are reranked by a cross-encoder that jointly reads the query and each passage. "
            "Reranking is slower than retrieval but usually improves precision at the top of the list. "
            "The final prompt keeps only the best 3 to 5 passages after reranking."
        ),
    },
    {
        "doc_id": "DOC-005",
        "title": "Freshness and deletion policy",
        "team": "ingestion",
        "updated_at": "2026-03-01",
        "text": (
            "Modified documents are re-chunked and re-embedded within 10 minutes. "
            "Deleted source files are tombstoned immediately and removed from the serving index within 15 minutes. "
            "This deletion path matters for privacy and compliance because stale chunks must not stay searchable. "
            "Embedding migrations run in the background and old and new versions are tracked side by side during rollout."
        ),
    },
    {
        "doc_id": "DOC-006",
        "title": "Generation policy",
        "team": "application",
        "updated_at": "2026-03-02",
        "text": (
            "The generator is instructed to answer only from retrieved evidence. "
            "If the evidence is insufficient, it must say that it does not have enough support in the retrieved notes. "
            "Answers should be short, include citations, and avoid blending retrieved facts with unsupported world knowledge. "
            "Prompt assembly also removes near-duplicate passages so the context window is not wasted."
        ),
    },
    {
        "doc_id": "DOC-007",
        "title": "Evaluation checklist",
        "team": "evaluation",
        "updated_at": "2026-03-05",
        "text": (
            "The team measures retrieval recall at 10, reranked MRR, groundedness, answer accuracy, latency, and cost. "
            "A good generator cannot rescue bad retrieval, so retrieval metrics are tracked separately from generation metrics. "
            "The current service target is p95 latency below 700 milliseconds for retrieval and below 2.5 seconds end to end. "
            "Every release is tested on a small labeled question set before deployment."
        ),
    },
    {
        "doc_id": "DOC-008",
        "title": "Security and access control",
        "team": "security",
        "updated_at": "2026-03-07",
        "text": (
            "Each chunk carries an acl field and retrieval must apply metadata filters before prompt assembly. "
            "This matters because the generator cannot be trusted to enforce permissions after seeing the text. "
            "For sensitive collections, the search service also logs which chunks were retrieved for each answer. "
            "Security bugs in RAG are often retrieval bugs, not generation bugs."
        ),
    },
]

docs_df = pd.DataFrame(documents)
docs_df[["doc_id", "title", "team", "updated_at"]]


## Chunking

Intro RAG lectures often rush through chunking.
In practice, chunking is one of the highest leverage choices in the entire system.

For this class:
- we will use a simple word-based chunker
- in production, chunk by document structure and tokenizer boundaries


In [ ]:
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = end - overlap

    return chunks


rows = []
for doc in documents:
    for idx, chunk in enumerate(chunk_text(doc["text"], chunk_size=55, overlap=10), start=1):
        rows.append(
            {
                "chunk_id": f'{doc["doc_id"]}-CH{idx:02d}',
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "team": doc["team"],
                "updated_at": doc["updated_at"],
                "chunk_text": chunk,
            }
        )

chunks_df = pd.DataFrame(rows)
print("documents:", len(docs_df))
print("chunks:", len(chunks_df))
chunks_df.head(8)


## Vector databases

A vector DB usually gives us:
- embedding storage
- ANN index structures such as HNSW or IVF
- metadata filters
- upsert and delete APIs
- persistence and scaling

Today we will simulate the vector store with a Pandas table plus cosine similarity.
Concept first, infrastructure later.


In [ ]:
embedder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embedder_name, device=device)

chunk_embeddings = embedder.encode(
    chunks_df["chunk_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("embedding model:", embedder_name)
print("embedding matrix:", chunk_embeddings.shape)


## Retrieval techniques

Retrieval is a full topic by itself.
Later we can cover sparse retrieval, BM25, hybrid search, query expansion, and learned retrievers.

In this notebook we keep only:
- dense embeddings
- cosine similarity
- optional reranking


In [ ]:
def semantic_search(query, k=5, filters=None):
    query_embedding = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    scores = chunk_embeddings @ query_embedding

    mask = np.ones(len(chunks_df), dtype=bool)
    if filters:
        for key, value in filters.items():
            mask &= chunks_df[key].eq(value).to_numpy()

    candidate_idx = np.where(mask)[0]
    ranked_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1][:k]]

    results = chunks_df.iloc[ranked_idx].copy()
    results["retrieval_score"] = scores[ranked_idx]
    return results.reset_index(drop=True)


query = "Why do we use a vector database instead of a plain NumPy matrix?"
semantic_search(query, k=4)[["chunk_id", "title", "team", "retrieval_score", "chunk_text"]]


In [ ]:
query = "How fast are deleted files removed from the serving index?"
semantic_search(query, k=4)[["chunk_id", "title", "updated_at", "retrieval_score"]]


In [ ]:
query = "Which team owns the reranker?"
semantic_search(query, k=4, filters={"team": "search"})[
    ["chunk_id", "title", "team", "retrieval_score"]
]


## Reranking

Dense retrieval is good for recall.
Reranking is often where precision improves.

Common pattern:
1. retrieve top 20 with a fast bi-encoder
2. rerank those 20 with a slower cross-encoder
3. send only the best few passages to the generator


In [ ]:
reranker_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(reranker_name, device=device)

def rerank(query, retrieved_df, top_n=3):
    pairs = [[query, chunk] for chunk in retrieved_df["chunk_text"].tolist()]
    scores = reranker.predict(pairs, show_progress_bar=False)

    reranked = retrieved_df.copy()
    reranked["rerank_score"] = scores
    reranked = reranked.sort_values("rerank_score", ascending=False).head(top_n)
    return reranked.reset_index(drop=True)


query = "What happens when a source document is deleted?"
retrieved = semantic_search(query, k=6)
reranked = rerank(query, retrieved, top_n=3)

print("before reranking")
display(retrieved[["chunk_id", "title", "retrieval_score"]])

print("after reranking")
display(reranked[["chunk_id", "title", "retrieval_score", "rerank_score"]])


## Generator

We now connect retrieval to a small local instruction model.

Model choice:
- `HuggingFaceTB/SmolLM2-360M-Instruct`
- small enough for classroom demos
- not state of the art, but good enough to demonstrate the RAG loop


In [ ]:
generator_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
generator_tokenizer = AutoTokenizer.from_pretrained(generator_name)
generator_model = AutoModelForCausalLM.from_pretrained(
    generator_name,
    torch_dtype=dtype,
).to(device)

print("generator:", generator_name)


In [ ]:
def format_context(passages):
    blocks = []
    for row in passages.itertuples(index=False):
        blocks.append(
            f'[{row.chunk_id}] {row.title} | team={row.team} | updated_at={row.updated_at}\n{row.chunk_text}'
        )
    return "\n\n".join(blocks)


def generate_answer(messages, max_new_tokens=180):
    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = generator_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = generator_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=generator_tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return generator_tokenizer.decode(new_tokens, skip_special_tokens=True)


def answer_without_rag(query):
    messages = [
        {"role": "system", "content": "Answer briefly and clearly."},
        {"role": "user", "content": query},
    ]
    return generate_answer(messages)


def answer_with_rag(query, retrieve_k=6, use_reranker=True, final_k=3):
    retrieved = semantic_search(query, k=retrieve_k)
    if use_reranker:
        passages = rerank(query, retrieved, top_n=final_k)
    else:
        passages = retrieved.head(final_k).copy()

    context = format_context(passages)

    messages = [
        {
            "role": "system",
            "content": (
                "Answer only from the retrieved notes. "
                "If the notes are insufficient, say so. "
                "Keep the answer short and cite chunk ids like [DOC-001-CH01]."
            ),
        },
        {
            "role": "user",
            "content": f"Question: {query}\n\nRetrieved notes:\n{context}",
        },
    ]
    answer = generate_answer(messages)
    return answer, passages


In [ ]:
question = "How quickly are deleted source files removed from the serving index?"

print("without RAG")
print(answer_without_rag(question))


In [ ]:
answer, passages = answer_with_rag(question)

print("with RAG")
print(answer)
print()
display(passages[["chunk_id", "title", "rerank_score", "chunk_text"]])


In [ ]:
question = "Why can't we trust the generator to enforce permissions after retrieval?"
answer, passages = answer_with_rag(question)

print(answer)
print()
display(passages[["chunk_id", "title", "team", "chunk_text"]])


## Long context vs retrieved context

In 2026, long context helps a lot.
But smaller and cleaner context is still useful.


In [ ]:
full_corpus = "\n\n".join(chunks_df["chunk_text"].tolist())
retrieved_context = format_context(passages)

full_tokens = len(generator_tokenizer.encode(full_corpus))
retrieved_tokens = len(generator_tokenizer.encode(retrieved_context))

print("tokens if we dump every chunk:", full_tokens)
print("tokens after retrieval + reranking:", retrieved_tokens)


## Things introductory RAG lectures often miss

- chunking strategy matters more than people expect
- metadata and ACL filters are part of retrieval quality
- reranking often helps more than changing the generator
- freshness, deletes, and embedding versioning are operational requirements
- duplicate removal improves prompt quality
- evaluation must separate retrieval quality from generation quality
- good RAG systems abstain when evidence is weak


## What a production RAG system usually adds

- document parsers for PDFs, HTML, tables, and code
- a real vector DB such as Qdrant, Weaviate, Milvus, Pinecone, or pgvector
- hybrid retrieval and query rewriting
- caching, observability, and tracing
- offline eval sets and online feedback loops
- tenant isolation and permission-aware search
- background jobs for updates, deletes, and re-embedding


## Limits of this notebook

This is still a teaching demo.

We simplified:
- chunking
- indexing
- retrieval scale
- prompt safety
- evaluation

But the overall shape is very close to a real production RAG pipeline.
